In [1]:
from pathlib import Path
from src.db import get_connection
import sqlite3

import numpy as np
import pandas as pd

import torch

print("cuda available" if torch.cuda.is_available() else "cuda unavailable")
print("gpu ready" if torch.cuda.device_count() else "only cpu")


DATABASE_PATH = Path.home() / "HDD/Datasets/annas_archive_spotify_2025_07/spotify_clean_playlists.sqlite3"

cuda available
gpu ready


In [2]:
conn  = get_connection(DATABASE_PATH)
print("Connected to database")

Connected to database


In [3]:
def fetch_topn_playlist_tracks(conn, n):
    """Fetch all valid (playlist_rowid, track_rowid) pairs for the top N playlists
    by followers. DISTINCT ensures each track appears at most once per playlist.
    """
    query = """
        SELECT DISTINCT pt.playlist_rowid, pt.track_rowid
        FROM playlist_tracks pt
        WHERE pt.playlist_rowid IN (
            SELECT rowid 
            FROM playlists 
            ORDER BY followers_total 
            DESC LIMIT ?
        )
          AND pt.is_episode = 0
          AND pt.is_local = 0
          AND pt.track_rowid IS NOT NULL
    """
    return pd.read_sql(query, conn, params=[n])

N = 32768
_playlist_tracks = fetch_topn_playlist_tracks(conn, N)
_playlist_tracks

,playlist_rowid,track_rowid
0,3,1
1,3,4
2,3,8
3,3,9
4,3,10
...,...,...
4343499,9882401,45162248
4343500,9882401,45162240
4343501,9882401,149605019
4343502,9882401,45162245


In [4]:
def filter_kcore(pt: pd.DataFrame, k: int, min_playlist_len: int=2) -> pd.DataFrame:
    """
    Iteratively remove tracks appearing in fewer than k playlists and playlists
    with fewer than min_playlist_len tracks, until the result is stable.

    A playlist with only one remaining track produces no center-context pairs
    and is useless for skip-gram training, hence min_playlist_len=2 by default.
    """
    while True:
        n_before = len(pt)

        track_counts = pt["track_rowid"].value_counts()
        pt = pt[pt["track_rowid"].map(track_counts) >= k]

        playlist_counts = pt["playlist_rowid"].value_counts()
        pt = pt[pt["playlist_rowid"].map(playlist_counts) >= min_playlist_len]

        if len(pt) == n_before:
            break
    return pt.reset_index(drop=True)


playlist_tracks = filter_kcore(_playlist_tracks, k=2)
print(f"Interactions    : {len(playlist_tracks):,}")
print(f"Unique tracks   : {playlist_tracks['track_rowid'].nunique():,}")
print(f"Playlists       : {playlist_tracks['playlist_rowid'].nunique():,}")
print(f"Min playlist len: {playlist_tracks["playlist_rowid"].value_counts().min()}")
print(f"Min track it.   : {playlist_tracks["track_rowid"].value_counts().min()}")

Interactions    : 3,025,875
Unique tracks   : 596,325
Playlists       : 32,292
Min playlist len: 2
Min track it.   : 2


In [5]:
def build_vocab(pt: pd.DataFrame) -> pd.DataFrame:
    """Build a track_rowid → track_id mapping from the filtered interaction table.

    Assigns a contiguous integer index to each unique track_rowid. The resulting
    DataFrame is indexed by track_rowid for fast join-based lookups.
    """
    unique_tracks = pt["track_rowid"].unique()
    return pd.DataFrame({
        "track_rowid": unique_tracks,
        "track_id": np.arange(len(unique_tracks)), 
    }).set_index("track_rowid")

vocab = build_vocab(playlist_tracks)
vocab

,track_id
track_rowid,
1,0
4,1
8,2
9,3
10,4
...,...
58565959,596320
99478108,596321
11762698,596322


In [6]:
def map_vocab(pt: pd.DataFrame, vocab: pd.DataFrame) -> pd.DataFrame:
    """Replace track_rowid with the contiguous track_id from the vocab."""
    return pt.join(vocab, on="track_rowid").drop(columns=["track_rowid"])

playlist_tracks = map_vocab(playlist_tracks, vocab)
playlist_tracks

,playlist_rowid,track_id
0,3,0
1,3,1
2,3,2
3,3,3
4,3,4
...,...,...
3025870,9882401,407568
3025871,9882401,407566
3025872,9882401,407565
3025873,9882401,596263


In [7]:
def build_weights(pt: pd.DataFrame) -> torch.Tensor:
    """Build a normalised negative-sampling weight tensor aligned to track_id order.

    Weights are proportional to freq^0.75 (word2vec unigram smoothing): frequent
    tracks are sampled more often as negatives, but the exponent dampens the
    dominance of the very top tracks.

    The returned tensor has shape (vocab_size,) where position i is the sampling
    weight for track_id=i, ready for torch.multinomial.
    """
    freq = pt["track_id"].value_counts()
    weights = torch.tensor((freq ** 0.75).sort_index().to_numpy(), dtype=torch.float32)
    return weights / weights.sum()

weights = build_weights(playlist_tracks)
weights

tensor([2.2672e-05, 2.3973e-06, 1.7616e-05,  ..., 9.3686e-07, 9.3686e-07,
        9.3686e-07])

In [8]:
def flatten(pt: pd.DataFrame) -> dict[int, list[int]]:
    """Group interactions into a playlist → [track_ids] dictionary."""
    return pt.groupby("playlist_rowid")["track_id"].apply(list).to_dict()

# Dataset

In [9]:
import random
import itertools
from warnings import warn
from typing import Iterable, Callable
from torch.utils.data import IterableDataset


def shuffle_buffer(iterable: Iterable, buffer_size: int):
    buf = []
    it = iter(iterable)

    # fill the buffer
    for item in itertools.islice(it, buffer_size):
        buf.append(item)

    # yield random elements, replacing with new ones from the stream
    for item in it:
        idx = random.randrange(len(buf))
        yield buf[idx]
        buf[idx] = item

    # drain whatever's left
    random.shuffle(buf)
    yield from buf


class SkipGramDataset(IterableDataset):
    def __init__(
        self,
        playlists_fn: Callable[[], Iterable[list[int]]],
        weights: torch.Tensor,
        W: int = 10,
        K: int = 5,
        sbuffer_size: int = 10_000,
    ):
        super().__init__()
        self.playlists_fn = playlists_fn
        self.weights = weights
        self.W = W
        self.K = K
        self.sbuffer_size = sbuffer_size
        self._epoch = 0

    def set_epoch(self, epoch: int) -> None:
        """Set the current epoch so that shuffling varies across epochs."""
        self._epoch = epoch

    def __iter__(self):
        worker_info = torch.utils.data.get_worker_info()

        if worker_info is None:
            worker_gen = self.playlists_fn()
            seed = self._epoch
        else:
            # Combine the per-worker seed with the epoch so that:
            #   - different workers get different orderings (worker_info.seed varies per worker)
            #   - the same worker gets different orderings across epochs (_epoch varies)
            seed = worker_info.seed + self._epoch
            # This is the generator equivalent to `playlists[worker_info.id :: worker_info.num_workers]`
            worker_gen = itertools.islice(self.playlists_fn(), worker_info.id, None, worker_info.num_workers)

        # Seed both Python's random (used for shuffle) and PyTorch's RNG (used for multinomial)
        random.seed(seed)
        torch.manual_seed(seed)
        
        for playlist in shuffle_buffer(worker_gen, buffer_size=self.sbuffer_size):
            if len(playlist) < 2:
                warn("Encountered a playlist with fewer than two elements. Skipping.")
                continue
        
            n_pairs = 0
            for i in range(len(playlist)):
                lo = max(0, i - self.W)
                hi = min(len(playlist), i + self.W + 1)
                n_pairs += hi - lo - 1  # exclude i == j
            all_negatives = torch.multinomial(
                self.weights, n_pairs * self.K, replacement=True
            ).view(n_pairs, self.K)
        
            idx = 0
            for i, center in enumerate(playlist):
                lo = max(0, i - self.W)
                hi = min(len(playlist), i + self.W + 1)
                for j in range(lo, hi):
                    if j == i:
                        continue
                    context = playlist[j]
                    yield center, context, all_negatives[idx]
                    idx += 1

# Model

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class Word2Vec(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int):
        super().__init__()
        self.embeddings_in = nn.Embedding(
            num_embeddings=vocab_size, embedding_dim=embed_dim, sparse=True
        )
        self.embeddings_out = nn.Embedding(
            num_embeddings=vocab_size, embedding_dim=embed_dim, sparse=True
        )
        nn.init.uniform_(self.embeddings_in.weight, -0.5 / embed_dim, 0.5 / embed_dim)
        nn.init.uniform_(self.embeddings_out.weight, -0.5 / embed_dim, 0.5 / embed_dim)

    def forward(
        self, center: torch.Tensor, context: torch.Tensor, negatives: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Returns:
            pos_score: (batch,) dot product between center and context embeddings
            neg_score: (batch, K) dot products between center and each negative embedding
        """
        ecenter = self.embeddings_in(center)
        econtext = self.embeddings_out(context)
        enegative = self.embeddings_out(negatives)

        pos_score = (ecenter * econtext).sum(dim=1)
        neg_score = torch.bmm(enegative, ecenter.unsqueeze(2)).squeeze(2)

        return pos_score, neg_score

    @property
    def track_embeddings(self) -> torch.Tensor:
        return self.embeddings_in.weight.detach()

# Loss

In [11]:
def skipgram_loss(
    pos_score: torch.Tensor, neg_score: torch.Tensor
) -> torch.Tensor:
    """Negative-sampling loss for skip-gram Word2Vec."""
    pos_loss = F.logsigmoid(pos_score)
    neg_loss = F.logsigmoid(-neg_score).sum(dim=1)
    return -(pos_loss + neg_loss).mean()

# Training

In [12]:
np.random.seed(0)

playlist_unique = playlist_tracks["playlist_rowid"].unique()
np.random.shuffle(playlist_unique)
playlist_train = playlist_unique[:int(0.9 * len(playlist_unique))]
train_mask = playlist_tracks["playlist_rowid"].isin(playlist_train)
train_df = playlist_tracks[train_mask].reset_index(drop=True)
valid_df = playlist_tracks[~train_mask].reset_index(drop=True)

print(f"train amounts to {100 * len(train_df) / len(playlist_tracks):.1f} % of the dataset")

train amounts to 90.1 % of the dataset


In [13]:
from torch.utils.data import DataLoader

train_dataset = SkipGramDataset(
    playlists_fn=lambda: [*flatten(train_df).values()],
    W=5,
    weights=weights,
    sbuffer_size=len(flatten(train_df)),
)
valid_dataset = SkipGramDataset(
    playlists_fn=lambda: [*flatten(valid_df).values()],
    W=5,
    weights=weights,
    sbuffer_size=len(flatten(valid_df)),
)
train = DataLoader(train_dataset, batch_size=512, num_workers=20, pin_memory=True, persistent_workers=True)
valid = DataLoader(valid_dataset, batch_size=512, num_workers=20, pin_memory=True, persistent_workers=True)

In [14]:
import time
from torch.optim import SparseAdam

EMBED_DIM = 128
NEPOCHS = 10
LR = 1e-3
device = torch.device("cuda")

model = Word2Vec(vocab_size=len(vocab), embed_dim=EMBED_DIM)
model = model.to(device)
optimizer = SparseAdam(model.parameters(), lr=LR)

history = {"train": [], "valid": []}

for epoch in range(NEPOCHS):
    t0 = time.perf_counter()

    model.train()
    train_dataset.set_epoch(epoch)
    nt, tloss_epoch = 0, 0.
    for nt, batch in enumerate(train):
        optimizer.zero_grad()
        batch = [x.to(device) for x in batch]
        pos, neg = model(*batch)
        loss = skipgram_loss(pos, neg)
        loss.backward()
        optimizer.step()
        tloss_epoch += loss.item()

    t1 = time.perf_counter()

    model.eval()
    valid_dataset.set_epoch(epoch)
    nv, vloss_epoch = 0, 0.
    with torch.no_grad():
        for nv, batch in enumerate(valid):
            batch = [x.to(device) for x in batch]
            vloss_epoch += skipgram_loss(*model(*batch)).item()

    t2 = time.perf_counter()

    train_loss = tloss_epoch / nt
    valid_loss = vloss_epoch / nv
    history["train"].append(train_loss)
    history["valid"].append(valid_loss)

    w = len(str(NEPOCHS))
    print(
        f"epoch {epoch+1:{w}}/{NEPOCHS}"
        f"  │  train {train_loss:.4f}"
        f"  valid {valid_loss:.4f}"
        f"  │  {t2-t0:.0f}s  (train {t1-t0:.0f}s  valid {t2-t1:.0f}s)"
    )

epoch  1/10  │  train 3.5916  valid 3.1448  │  118s  (train 111s  valid 7s)
epoch  2/10  │  train 2.7497  valid 2.7988  │  115s  (train 109s  valid 6s)
epoch  3/10  │  train 2.3994  valid 2.6185  │  115s  (train 109s  valid 6s)
epoch  4/10  │  train 2.1935  valid 2.4940  │  116s  (train 110s  valid 6s)
epoch  5/10  │  train 2.0484  valid 2.3961  │  119s  (train 112s  valid 6s)
epoch  6/10  │  train 1.9315  valid 2.3149  │  119s  (train 112s  valid 6s)
epoch  7/10  │  train 1.8285  valid 2.2457  │  116s  (train 110s  valid 6s)
epoch  8/10  │  train 1.7333  valid 2.1859  │  112s  (train 106s  valid 6s)
epoch  9/10  │  train 1.6436  valid 2.1336  │  112s  (train 106s  valid 6s)
epoch 10/10  │  train 1.5588  valid 2.0869  │  125s  (train 119s  valid 6s)


### Recap

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(4, 4))
epochs = range(1, len(history["train"]) + 1)
ax.plot(epochs, history["train"], label="train")
ax.plot(epochs, history["valid"], label="valid")
ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.set_title("SGNS loss"); ax.legend()
plt.show()